In [3]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [4]:
# Q8: TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LOW PRESSURE SWITCH (LPS) CAO NHẤT
q8 = spark.sql('''
    SELECT
        date,
        COUNT(*) AS tong_so_ban_ghi,
        SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) AS so_lan_lps_kich_hoat,
        ROUND(
            SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
            2
        ) AS ty_le_lps_kich_hoat_pct
    FROM sensor
    GROUP BY date
    HAVING SUM(CASE WHEN CAST(LPS AS STRING) = '1' THEN 1 ELSE 0 END) > 0
    ORDER BY so_lan_lps_kich_hoat DESC
    LIMIT 10
''')

# [GIỮ NGUYÊN] In kế hoạch thực thi chi tiết cho báo cáo
print("\n========== EXECUTION PLAN ==========")
q8.explain(True)

# [GIỮ NGUYÊN] Chuyển đổi sang Pandas sau khi giải thích xong
print("\n--- TOP 10 NGÀY CÓ TẦN SUẤT KÍCH HOẠT LPS CAO NHẤT ---")
df_q8 = q8.toPandas()

# Phần hiển thị: Giữ nguyên logic của bạn nhưng thêm bẫy kiểm tra hiển thị
if df_q8.empty:
    print("Mẹo: Bảng kết quả trả về 0 dòng. Điều này có nghĩa là trong tập dữ liệu sensor hiện tại, không có bản ghi nào mà LPS có giá trị bằng 1.")
else:
    try:
        display(df_q8)
    except NameError:
        # Ép in trực tiếp ra console nếu hàm display() của môi trường không hoạt động
        print(df_q8.to_string(index=False))


========== EXECUTION PLAN ==========
== Parsed Logical Plan ==
'GlobalLimit 10
+- 'LocalLimit 10
   +- 'Sort ['so_lan_lps_kich_hoat DESC NULLS LAST], true
      +- 'UnresolvedHaving ('SUM(CASE WHEN (cast('LPS as string) = 1) THEN 1 ELSE 0 END) > 0)
         +- 'Aggregate ['date], ['date, 'COUNT(1) AS tong_so_ban_ghi#762, 'SUM(CASE WHEN (cast('LPS as string) = 1) THEN 1 ELSE 0 END) AS so_lan_lps_kich_hoat#763, 'ROUND((('SUM(CASE WHEN (cast('LPS as string) = 1) THEN 1 ELSE 0 END) * 100.0) / 'COUNT(1)), 2) AS ty_le_lps_kich_hoat_pct#764]
            +- 'UnresolvedRelation [sensor], [], false

== Analyzed Logical Plan ==
date: date, tong_so_ban_ghi: bigint, so_lan_lps_kich_hoat: bigint, ty_le_lps_kich_hoat_pct: decimal(27,2)
GlobalLimit 10
+- LocalLimit 10
   +- Sort [so_lan_lps_kich_hoat#763L DESC NULLS LAST], true
      +- Filter (so_lan_lps_kich_hoat#763L > cast(0 as bigint))
         +- Aggregate [date#75], [date#75, count(1) AS tong_so_ban_ghi#762L, sum(CASE WHEN (cast(LPS#64 as stri

,date,tong_so_ban_ghi,so_lan_lps_kich_hoat,ty_le_lps_kich_hoat_pct
0,2020-07-15,8661,534,6.17
1,2020-07-17,7279,414,5.69
2,2020-06-08,4232,312,7.37
3,2020-05-19,7410,272,3.67
4,2020-07-31,7119,247,3.47
5,2020-07-08,3107,238,7.66
6,2020-07-14,3563,224,6.29
7,2020-06-02,5131,203,3.96
8,2020-03-12,8202,170,2.07
9,2020-08-03,3947,169,4.28
